# 17 — External Test Evaluation (Sanity Check Only)

Mục tiêu: kiểm tra pipeline đã khóa trên một bộ ảnh bên ngoài do bạn tự thu thập.

**External test path**
`C:\Users\Admin\Downloads\test`

Expected structure:

```text
test/
├─ NORMAL/
│  ├─ ...
└─ PNEUMONIA/
   ├─ ...
```

Pipeline giữ nguyên:
- DenseNet121
- 224×224
- grayscale → RGB
- seed 2026 weight = 0.70
- seed 3407 weight = 0.30
- threshold = 0.576
- TTA = OFF

Notebook này **không được dùng để retune model**. Nó chỉ là external sanity check.


In [ ]:
print('>>> CELL 1 START: config', flush=True)
import os,time,gc,json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image,ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms,models
from sklearn.metrics import confusion_matrix,f1_score,roc_auc_score

ROOT=Path(r'D:\medicine')
EXTERNAL_ROOT=Path(r'C:\Users\Admin\Downloads\test')
LOCK_PATH=ROOT/'cv_outputs'/'final_ensemble_lock'/'locked_pipeline.json'
OUT=ROOT/'external_test_outputs'
OUT.mkdir(parents=True,exist_ok=True)

VALID_EXTENSIONS={'.png','.jpg','.jpeg','.bmp','.tif','.tiff'}
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
BATCH_SIZE=16

print('Device:',DEVICE,flush=True)
if USE_AMP: print('GPU:',torch.cuda.get_device_name(0),flush=True)
print('External root:',EXTERNAL_ROOT,flush=True)


In [ ]:
print('>>> CELL 2 START: load locked pipeline + discover labeled external data', flush=True)

with open(LOCK_PATH,'r',encoding='utf-8') as f:
    lock=json.load(f)

assert lock['status']=='LOCKED_BEFORE_HOLDOUT'
assert lock['architecture']=='DenseNet121'
assert lock['image_size']==224
assert lock['preprocessing']=='force_grayscale_then_rgb'
assert lock['tta'] is False

def image_paths(root):
    root=Path(root)
    return sorted(
        [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS],
        key=lambda p:str(p).lower()
    )

if not EXTERNAL_ROOT.exists():
    raise FileNotFoundError(f'External test folder not found: {EXTERNAL_ROOT}')

children={p.name.lower():p for p in EXTERNAL_ROOT.iterdir() if p.is_dir()}
normal_dir=children.get('normal')
pneumonia_dir=children.get('pneumonia')

if normal_dir is None or pneumonia_dir is None:
    raise RuntimeError(
        f'Expected NORMAL and PNEUMONIA subfolders under {EXTERNAL_ROOT}. Found: {[p.name for p in EXTERNAL_ROOT.iterdir() if p.is_dir()]}'
    )

normal_paths=image_paths(normal_dir)
pneumonia_paths=image_paths(pneumonia_dir)

if len(normal_paths)==0 or len(pneumonia_paths)==0:
    raise RuntimeError('NORMAL and PNEUMONIA folders must both contain images.')

samples=[(p,0) for p in normal_paths]+[(p,1) for p in pneumonia_paths]

print('Normal:',len(normal_paths),flush=True)
print('Pneumonia:',len(pneumonia_paths),flush=True)
print('Total:',len(samples),flush=True)
print('Threshold:',lock['threshold'],flush=True)
print('Weights:',[(m['name'],m['weight']) for m in lock['models']],flush=True)


In [ ]:
print('>>> CELL 3 START: dataset + helpers', flush=True)

IMAGE_SIZE=int(lock['image_size'])
THRESHOLD=float(lock['threshold'])
MEAN=[0.485,0.456,0.406]
STD=[0.229,0.224,0.225]

class PadToSquare:
    def __call__(self,img):
        w,h=img.size
        side=max(w,h)
        left=(side-w)//2; right=side-w-left
        top=(side-h)//2; bottom=side-h-top
        return ImageOps.expand(img,border=(left,top,right,bottom),fill=0)

eval_tf=transforms.Compose([
    PadToSquare(),
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE),interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])

class ExternalDataset(Dataset):
    def __init__(self,samples):
        self.samples=list(samples)
    def __len__(self): return len(self.samples)
    def __getitem__(self,i):
        p,y=self.samples[i]
        with Image.open(p) as im:
            x=eval_tf(im.convert('L').convert('RGB'))
        return x,int(y),str(p)

loader=DataLoader(
    ExternalDataset(samples),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

def make_model():
    m=models.densenet121(weights=None)
    m.classifier=nn.Linear(m.classifier.in_features,2)
    return m.to(DEVICE)

@torch.no_grad()
def infer_checkpoint(checkpoint_path):
    ck=torch.load(checkpoint_path,map_location='cpu')
    model=make_model()
    model.load_state_dict(ck['state_dict'],strict=True)
    model.eval()

    ys=[];ps=[];paths=[]

    for x,y,p in loader:
        x=x.to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
            logits=model(x)
        probs=torch.softmax(logits.float(),dim=1)[:,1]
        ys.extend(y.numpy())
        ps.extend(probs.cpu().numpy())
        paths.extend(list(p))

    del model,ck
    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()

    return np.asarray(ys),np.asarray(ps),paths

def metrics(y,p,t):
    y=np.asarray(y,int);p=np.asarray(p,float)
    pred=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    acc=(tp+tn)/len(y)
    se=tp/max(tp+fn,1)
    sp=tn/max(tn+fp,1)
    f1=f1_score(y,pred,pos_label=1,zero_division=0)
    P=.30*acc+.25*se+.20*sp+.25*f1
    auc=roc_auc_score(y,p)
    return {
        'P':float(P),
        'Accuracy':float(acc),
        'Sensitivity':float(se),
        'Specificity':float(sp),
        'F1':float(f1),
        'AUROC':float(auc),
        'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
        'threshold':float(t)
    }


In [ ]:
print('>>> CELL 4 START: locked ensemble inference', flush=True)

seed_probs={}
y_ref=None
paths_ref=None

for spec in lock['models']:
    name=spec['name']
    ckpt_dir=Path(spec['checkpoint_dir'])
    fold_probs=[]

    print('\nSeed group:',name,flush=True)

    for fold in range(5):
        ckpt=ckpt_dir/f'fold{fold}_best.pt'
        if not ckpt.exists():
            raise FileNotFoundError(f'Missing checkpoint: {ckpt}')

        t0=time.time()
        y,p,paths=infer_checkpoint(ckpt)

        if y_ref is None:
            y_ref=y
            paths_ref=paths
        else:
            assert np.array_equal(y_ref,y)
            assert paths_ref==paths

        fold_probs.append(p)
        print(f'  fold {fold}/4: {time.time()-t0:.1f}s',flush=True)

    seed_probs[name]=np.mean(np.stack(fold_probs,axis=1),axis=1)

final_prob=np.zeros(len(y_ref),dtype=float)
for spec in lock['models']:
    final_prob += float(spec['weight'])*seed_probs[spec['name']]

result_metrics=metrics(y_ref,final_prob,THRESHOLD)
pred=(final_prob>=THRESHOLD).astype(int)

print('\nLOCKED EXTERNAL TEST METRICS')
print(json.dumps(result_metrics,indent=2),flush=True)


In [ ]:
print('>>> CELL 5 START: save results + error table', flush=True)

label_names=np.array(['Normal','Pneumonia'])

rows=pd.DataFrame({
    'path':paths_ref,
    'filename':[Path(p).name for p in paths_ref],
    'true_label':label_names[y_ref],
    'p_Pneumonia':final_prob,
    'predicted_label':label_names[pred],
    'correct':pred==y_ref,
})

rows.to_csv(OUT/'external_test_predictions.csv',index=False)

errors=rows[~rows.correct].copy()
errors.to_csv(OUT/'external_test_errors.csv',index=False)

summary={
    'evaluation':'EXTERNAL_SANITY_CHECK_ONLY',
    'external_root':str(EXTERNAL_ROOT),
    'n_normal':int((y_ref==0).sum()),
    'n_pneumonia':int((y_ref==1).sum()),
    'n_total':int(len(y_ref)),
    'pipeline':{
        'architecture':'DenseNet121',
        'image_size':224,
        'preprocessing':'force_grayscale_then_rgb',
        'weights':{m['name']:float(m['weight']) for m in lock['models']},
        'threshold':THRESHOLD,
        'tta':False,
    },
    'metrics':result_metrics,
    'errors':int((pred!=y_ref).sum()),
    'DO_NOT_RETUNE_FROM_EXTERNAL_TEST':True
}

with open(OUT/'external_test_summary.json','w',encoding='utf-8') as f:
    json.dump(summary,f,indent=2)

print('\nSUMMARY')
print(json.dumps(summary,indent=2),flush=True)
print('\nFirst errors:')
display(errors.head(20))


## Cách đọc kết quả

Metric quan trọng:
- **P** — điểm theo công thức cuộc thi
- **Sensitivity** — bắt đúng Pneumonia
- **Specificity** — nhận đúng Normal
- **F1**
- **AUROC**
- **TN / FP / FN / TP**

Nếu điểm external thấp hơn sealed holdout, điều đó có thể phản ánh domain shift giữa nguồn ảnh ngoài và dataset cuộc thi. Không được dùng external test này để thay đổi threshold/weight của pipeline đã khóa.
